# Zivilluftfahrt Datenanalyse - Mini-Projekt

## Inhaltsverzeichnis
1. [Team](#team)
2. [Daten](#daten)
3. [Datenbereinigung](#datenbereinigung)
4. [Datenvorbereitung](#datenvorbereitung)
5. [Visualisierung](#visualisierung)

## 1. Team
Team-Mitglieder: [Team-Namen hier einfügen]

## 2. Daten

### Datensatz-Link
Quelle: https://data.statistik.gv.at/web/catalog.jsp

### Beschreibung der Attribute
- **C-ZLFMONAT-0**: Monat im Format YYYYMM (ordinalskaliert)
- **C-ZLFE32-0**: Flughafen-Code (VIE=Wien, GRZ=Graz, INN=Innsbruck, KLU=Klagenfurt, LNZ=Linz, SZG=Salzburg) (nominal)
- **C-ZLFART-0**: Art der Bewegung (1=Linienverkehr, 2=Charterverkehr) (nominal)
- **C-ZLFR63-0**: Richtung (1=Abflug, 2=Ankunft, 3=Transit) (nominal)
- **F-ZLFFLUG**: Anzahl der Flüge (verhältnisskaliert)
- **F-ZLFPAX**: Anzahl der Passagiere (verhältnisskaliert)
- **F-ZLFFRACHT**: Fracht in kg (verhältnisskaliert)
- **F-ZLFPOST**: Post in kg (verhältnisskaliert)
- **F-ZLFPAXEND**: Endgültige Passagiere (verhältnisskaliert)

In [16]:
# Bibliotheken importieren
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Visualisierung konfigurieren
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

In [17]:
# Daten laden
df = pd.read_csv('./OGD_zlf_komm_ZLF_KOM_1.csv')
print(f"Datensatz geladen mit {len(df)} Zeilen und {len(df.columns)} Spalten")

Datensatz geladen mit 9899 Zeilen und 1 Spalten


### Initiale Standard-Analyse

In [10]:
# Sample - Stichprobe der Daten
print("=== SAMPLE - Zufällige Stichprobe ===")
df.sample(10)

=== SAMPLE - Zufällige Stichprobe ===


,C-ZLFMONAT-0;C-ZLFE32-0;C-ZLFART-0;C-ZLFR63-0;F-ZLFFLUG;F-ZLFPAX;F-ZLFFRACHT;F-ZLFPOST;F-ZLFPAXEND
4730,201107;KLU;2;1;56;1666;0;0;0
3592,200809;INN;2;3;0;2111;0;0;0
8636,202204;SZG;1;2;356;35554;3394;0;35554
3659,200811;INN;2;3;0;79;0;0;0
2173,200503;SZG;1;2;811;48411;5729;46;48411
908,200203;VIE;2;2;421;32167;243453;23;32167
3421,200804;KLU;1;1;289;14124;1611;0;0
9048,202306;LNZ;1;1;49;3045;0;0;0
6258,201507;KLU;1;1;168;11895;0;0;0
7684,201905;VIE;1;2;11896;1408863;7444045;554401;14...


In [11]:
# Head - Erste Zeilen
print("=== HEAD - Erste 10 Zeilen ===")
df.head(10)

=== HEAD - Erste 10 Zeilen ===


,C-ZLFMONAT-0;C-ZLFE32-0;C-ZLFART-0;C-ZLFR63-0;F-ZLFFLUG;F-ZLFPAX;F-ZLFFRACHT;F-ZLFPOST;F-ZLFPAXEND
0,200001;VIE;1;1;6351;316670;3743524;219204;0
1,200001;VIE;1;2;6376;325509;4000620;365309;325509
2,200001;VIE;1;3;0;6634;150274;14860;0
3,200001;VIE;2;1;355;20503;463078;68;0
4,200001;VIE;2;2;346;20965;412342;61;20965
5,200001;VIE;2;3;0;2302;136417;472;0
6,200001;GRZ;1;1;464;13887;66281;198;0
7,200001;GRZ;1;2;447;14327;44842;0;14327
8,200001;GRZ;1;3;0;284;25;0;0
9,200001;GRZ;2;1;50;2504;0;0;0


In [12]:
# Info - Datentypen und Speicher
print("=== INFO - Datenstruktur ===")
df.info()

=== INFO - Datenstruktur ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9899 entries, 0 to 9898
Data columns (total 1 columns):
 #   Column                                                                                              Non-Null Count  Dtype 
---  ------                                                                                              --------------  ----- 
 0   C-ZLFMONAT-0;C-ZLFE32-0;C-ZLFART-0;C-ZLFR63-0;F-ZLFFLUG;F-ZLFPAX;F-ZLFFRACHT;F-ZLFPOST;F-ZLFPAXEND  9899 non-null   object
dtypes: object(1)
memory usage: 77.5+ KB


In [13]:
# Describe - Statistische Kennzahlen
print("=== DESCRIBE - Statistische Kennzahlen ===")
df.describe()

=== DESCRIBE - Statistische Kennzahlen ===


,C-ZLFMONAT-0;C-ZLFE32-0;C-ZLFART-0;C-ZLFR63-0;F-ZLFFLUG;F-ZLFPAX;F-ZLFFRACHT;F-ZLFPOST;F-ZLFPAXEND
count,9899
unique,9899
top,200001;VIE;1;1;6351;316670;3743524;219204;0
freq,1


In [14]:
# Unique Werte für kategorische Variablen
print("=== UNIQUE WERTE ===")
print(f"\nFlughäfen (C-ZLFE32-0): {df['C-ZLFE32-0'].unique()}")
print(f"Anzahl: {df['C-ZLFE32-0'].nunique()}")

print(f"\nVerkehrsart (C-ZLFART-0): {df['C-ZLFART-0'].unique()}")
print(f"Anzahl: {df['C-ZLFART-0'].nunique()}")

print(f"\nRichtung (C-ZLFR63-0): {df['C-ZLFR63-0'].unique()}")
print(f"Anzahl: {df['C-ZLFR63-0'].nunique()}")

print(f"\nMonatswerte (C-ZLFMONAT-0): Min={df['C-ZLFMONAT-0'].min()}, Max={df['C-ZLFMONAT-0'].max()}")
print(f"Anzahl verschiedener Monate: {df['C-ZLFMONAT-0'].nunique()}")

=== UNIQUE WERTE ===


KeyError: 'C-ZLFE32-0'

## 3. Datenbereinigung

In diesem Abschnitt werden die Daten auf Qualität geprüft und bereinigt.

In [18]:
# Fehlende Werte analysieren
print("=== FEHLENDE WERTE ===")
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Fehlend': missing, 'Prozent': missing_percent})
print(missing_df[missing_df['Fehlend'] > 0])

if missing.sum() == 0:
    print("Keine fehlenden Werte gefunden!")

=== FEHLENDE WERTE ===
Empty DataFrame
Columns: [Fehlend, Prozent]
Index: []
Keine fehlenden Werte gefunden!


In [19]:
# Falsche/ungültige Werte prüfen
print("=== PRÜFUNG AUF UNGÜLTIGE WERTE ===")

# Negative Werte bei Zählvariablen
numeric_cols = ['F-ZLFFLUG', 'F-ZLFPAX', 'F-ZLFFRACHT', 'F-ZLFPOST', 'F-ZLFPAXEND']
for col in numeric_cols:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"WARNUNG: {col} hat {negative_count} negative Werte")
    else:
        print(f"OK: {col} hat keine negativen Werte")

# Duplikate prüfen
duplicates = df.duplicated().sum()
print(f"\nAnzahl Duplikate: {duplicates}")

=== PRÜFUNG AUF UNGÜLTIGE WERTE ===


KeyError: 'F-ZLFFLUG'

In [ ]:
# Feature-Reduktion und Typsetzung
print("=== FEATURE-REDUKTION UND TYPSETZUNG ===")

# Datenkopie für Bereinigung erstellen
df_clean = df.copy()

# Kategorische Variablen als Category-Typ setzen
df_clean['C-ZLFE32-0'] = df_clean['C-ZLFE32-0'].astype('category')
df_clean['C-ZLFART-0'] = df_clean['C-ZLFART-0'].astype('category')
df_clean['C-ZLFR63-0'] = df_clean['C-ZLFR63-0'].astype('category')

# Monat als String behandeln für bessere Analyse
df_clean['C-ZLFMONAT-0'] = df_clean['C-ZLFMONAT-0'].astype(str)

# Jahr und Monat extrahieren für spätere Analysen
df_clean['Jahr'] = df_clean['C-ZLFMONAT-0'].str[:4].astype(int)
df_clean['Monat'] = df_clean['C-ZLFMONAT-0'].str[4:6].astype(int)

print("Datentypen nach Bereinigung:")
print(df_clean.dtypes)

print(f"\nBereinigte Daten: {len(df_clean)} Zeilen")

In [ ]:
# Bereinigte Daten speichern
df_clean.to_csv('korr.csv', index=False)
print("Bereinigte Daten in 'korr.csv' gespeichert")

## 4. Datenvorbereitung

Aufbereitung der Daten für verschiedene Analyseformen.

In [ ]:
# Numerische Werte extrahieren
print("=== NUMERISCHE DATEN ===")

numeric_columns = ['F-ZLFFLUG', 'F-ZLFPAX', 'F-ZLFFRACHT', 'F-ZLFPOST', 'F-ZLFPAXEND', 'Jahr', 'Monat']
df_numeric = df_clean[numeric_columns].copy()

print(f"Numerischer Datensatz: {df_numeric.shape}")
print("\nStatistische Kennzahlen:")
df_numeric.describe()

In [ ]:
# Numerische Daten speichern
df_numeric.to_csv('num.csv', index=False)
print("Numerische Daten in 'num.csv' gespeichert")

In [ ]:
# Normalisierte Werte (Min-Max Normalisierung auf [0,1])
print("=== NORMALISIERTE DATEN ===")

scaler = MinMaxScaler()
df_normalized = pd.DataFrame(
    scaler.fit_transform(df_numeric),
    columns=df_numeric.columns
)

print(f"Normalisierter Datensatz: {df_normalized.shape}")
print("\nStatistische Kennzahlen (normalisiert):")
df_normalized.describe()

In [ ]:
# Normalisierte Daten speichern
df_normalized.to_csv('norm.csv', index=False)
print("Normalisierte Daten in 'norm.csv' gespeichert")

In [ ]:
# Nominale Werte
print("=== NOMINALE DATEN ===")

nominal_columns = ['C-ZLFMONAT-0', 'C-ZLFE32-0', 'C-ZLFART-0', 'C-ZLFR63-0']
df_nominal = df_clean[nominal_columns].copy()

print(f"Nominaler Datensatz: {df_nominal.shape}")
print("\nWertehäufigkeiten:")
for col in nominal_columns:
    print(f"\n{col}:")
    print(df_nominal[col].value_counts().head())

In [ ]:
# Nominale Daten speichern
df_nominal.to_csv('nom.csv', index=False)
print("Nominale Daten in 'nom.csv' gespeichert")

## 5. Visualisierung mit Seaborn

Umfassende Visualisierung der Daten mit verschiedenen Diagrammtypen.

In [ ]:
# Eigene Farbpalette definieren
custom_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F']
sns.set_palette(custom_colors)

print("Eigene Farbpalette definiert")
sns.palplot(sns.color_palette(custom_colors))
plt.title('Eigene Farbpalette')
plt.show()

In [ ]:
# Korrelationsmatrix mit matplotlib.pyplot.matshow()
print("=== KORRELATION - Matplotlib matshow() ===")

correlation = df_numeric.corr()

fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.matshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(cax)

# Achsenbeschriftungen
ax.set_xticks(range(len(correlation.columns)))
ax.set_yticks(range(len(correlation.columns)))
ax.set_xticklabels(correlation.columns, rotation=90)
ax.set_yticklabels(correlation.columns)

plt.title('Korrelationsmatrix (Matplotlib matshow)', pad=20)
plt.tight_layout()
plt.show()

print("\nAuffälligkeiten:")
print("- Starke Korrelation zwischen F-ZLFPAX und F-ZLFPAXEND (erwartbar, da eng verwandt)")
print("- Moderate Korrelation zwischen Flügen und Passagieren")
print("- Fracht und Post zeigen schwächere Korrelationen zu Passagierzahlen")

In [ ]:
# Korrelationsmatrix mit seaborn.heatmap()
print("=== KORRELATION - Seaborn heatmap() ===")

plt.figure(figsize=(12, 10))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='RdYlBu_r', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Korrelationsmatrix (Seaborn heatmap)', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("\nAuffälligkeiten:")
print("- Die Heatmap zeigt die Korrelationswerte deutlicher mit Farbcodierung")
print("- Negative Korrelationen sind in Blautönen, positive in Rottönen dargestellt")
print("- Jahr zeigt teilweise negative Korrelationen, was auf Trends hindeutet")

In [ ]:
# Pairplot - Für bessere Performance nur Subset
print("=== PAIRPLOT ===")

# Auswahl wichtiger numerischer Variablen
pairplot_cols = ['F-ZLFFLUG', 'F-ZLFPAX', 'F-ZLFFRACHT', 'F-ZLFPOST']
df_pairplot = df_clean[pairplot_cols + ['C-ZLFE32-0']].copy()

# Sample für Performance (jeder 10. Eintrag)
df_pairplot_sample = df_pairplot.iloc[::10]

sns.pairplot(df_pairplot_sample, hue='C-ZLFE32-0', palette=custom_colors, 
             diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Pairplot - Beziehungen zwischen Variablen nach Flughafen', y=1.01)
plt.tight_layout()
plt.show()

print("\nAuffälligkeiten:")
print("- VIE (Wien) zeigt deutlich höhere Werte in allen Kategorien")
print("- Starke lineare Beziehung zwischen Flügen und Passagieren sichtbar")
print("- Fracht und Post zeigen weniger konsistente Muster")

In [ ]:
# Box- vs Violinplot mit Subplots
print("=== BOX- VS VIOLINPLOT ===")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Boxplot - Passagiere nach Flughafen
sns.boxplot(data=df_clean, x='C-ZLFE32-0', y='F-ZLFPAX', ax=axes[0, 0], palette=custom_colors)
axes[0, 0].set_title('Boxplot - Passagiere nach Flughafen', fontsize=12)
axes[0, 0].set_xlabel('Flughafen')
axes[0, 0].set_ylabel('Passagiere')

# Violinplot - Passagiere nach Flughafen
sns.violinplot(data=df_clean, x='C-ZLFE32-0', y='F-ZLFPAX', ax=axes[0, 1], palette=custom_colors)
axes[0, 1].set_title('Violinplot - Passagiere nach Flughafen', fontsize=12)
axes[0, 1].set_xlabel('Flughafen')
axes[0, 1].set_ylabel('Passagiere')

# Boxplot - Flüge nach Verkehrsart
sns.boxplot(data=df_clean, x='C-ZLFART-0', y='F-ZLFFLUG', ax=axes[1, 0], palette=custom_colors[:2])
axes[1, 0].set_title('Boxplot - Flüge nach Verkehrsart', fontsize=12)
axes[1, 0].set_xlabel('Verkehrsart (1=Linie, 2=Charter)')
axes[1, 0].set_ylabel('Anzahl Flüge')

# Violinplot - Flüge nach Verkehrsart
sns.violinplot(data=df_clean, x='C-ZLFART-0', y='F-ZLFFLUG', ax=axes[1, 1], palette=custom_colors[:2])
axes[1, 1].set_title('Violinplot - Flüge nach Verkehrsart', fontsize=12)
axes[1, 1].set_xlabel('Verkehrsart (1=Linie, 2=Charter)')
axes[1, 1].set_ylabel('Anzahl Flüge')

plt.tight_layout()
plt.show()

print("\nAuffälligkeiten:")
print("- Boxplots zeigen Median, Quartile und Ausreißer deutlich")
print("- Violinplots zeigen die Verteilungsdichte besser")
print("- VIE hat deutlich höhere Passagierzahlen und größere Variabilität")
print("- Linienverkehr zeigt mehr Flüge als Charterverkehr")
print("- Viele Ausreißer bei kleineren Flughäfen sichtbar")

In [ ]:
# Histogramme und KDE (Kernel Density Estimation)
print("=== HISTOGRAMME UND KDE ===")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Histogramm - Passagiere
axes[0, 0].hist(df_clean['F-ZLFPAX'], bins=50, color=custom_colors[0], alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Histogramm - Passagiere', fontsize=12)
axes[0, 0].set_xlabel('Passagiere')
axes[0, 0].set_ylabel('Häufigkeit')
axes[0, 0].grid(alpha=0.3)

# KDE - Passagiere
df_clean['F-ZLFPAX'].plot.kde(ax=axes[0, 1], color=custom_colors[1], linewidth=2)
axes[0, 1].set_title('KDE - Passagiere', fontsize=12)
axes[0, 1].set_xlabel('Passagiere')
axes[0, 1].set_ylabel('Dichte')
axes[0, 1].grid(alpha=0.3)

# Histogramm mit KDE - Fracht
sns.histplot(data=df_clean, x='F-ZLFFRACHT', kde=True, color=custom_colors[2], ax=axes[1, 0])
axes[1, 0].set_title('Histogramm mit KDE - Fracht', fontsize=12)
axes[1, 0].set_xlabel('Fracht (kg)')
axes[1, 0].set_ylabel('Häufigkeit')

# Mehrere KDE-Plots - Passagiere nach Verkehrsart
for art in df_clean['C-ZLFART-0'].unique():
    subset = df_clean[df_clean['C-ZLFART-0'] == art]
    subset['F-ZLFPAX'].plot.kde(ax=axes[1, 1], label=f"Art {art}", linewidth=2)
axes[1, 1].set_title('KDE - Passagiere nach Verkehrsart', fontsize=12)
axes[1, 1].set_xlabel('Passagiere')
axes[1, 1].set_ylabel('Dichte')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAuffälligkeiten:")
print("- Passagierzahlen zeigen eine rechtsschiefe Verteilung")
print("- Viele Flüge mit niedrigen Passagierzahlen, wenige mit sehr hohen")
print("- Fracht zeigt ähnliche Verteilung mit Konzentration bei niedrigen Werten")
print("- Linien- und Charterverkehr haben unterschiedliche Verteilungsprofile")

In [ ]:
# Scatterplots
print("=== SCATTERPLOTS ===")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter - Flüge vs Passagiere nach Flughafen
for i, airport in enumerate(df_clean['C-ZLFE32-0'].unique()):
    subset = df_clean[df_clean['C-ZLFE32-0'] == airport]
    axes[0].scatter(subset['F-ZLFFLUG'], subset['F-ZLFPAX'], 
                   label=airport, alpha=0.6, s=20, color=custom_colors[i % len(custom_colors)])
axes[0].set_title('Scatterplot - Flüge vs Passagiere nach Flughafen', fontsize=12)
axes[0].set_xlabel('Anzahl Flüge')
axes[0].set_ylabel('Anzahl Passagiere')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Scatter - Fracht vs Passagiere
sns.scatterplot(data=df_clean, x='F-ZLFFRACHT', y='F-ZLFPAX', 
               hue='C-ZLFART-0', palette=custom_colors, alpha=0.6, ax=axes[1])
axes[1].set_title('Scatterplot - Fracht vs Passagiere nach Verkehrsart', fontsize=12)
axes[1].set_xlabel('Fracht (kg)')
axes[1].set_ylabel('Anzahl Passagiere')
axes[1].legend(title='Verkehrsart')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAuffälligkeiten:")
print("- Klarer linearer Zusammenhang zwischen Flügen und Passagieren")
print("- VIE dominiert mit höchsten Werten in beiden Dimensionen")
print("- Fracht und Passagiere zeigen schwächeren Zusammenhang")
print("- Einige Flüge mit hoher Fracht aber wenigen Passagieren (Cargo-Flüge)")

In [ ]:
# Linien- und Balkendiagramme
print("=== LINIEN- UND BALKENDIAGRAMME ===")

# Daten aggregieren
passengers_by_airport = df_clean.groupby('C-ZLFE32-0')['F-ZLFPAX'].sum().sort_values(ascending=False)
passengers_by_year = df_clean.groupby('Jahr')['F-ZLFPAX'].sum()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Balkendiagramm - Passagiere nach Flughafen
passengers_by_airport.plot(kind='bar', ax=axes[0, 0], color=custom_colors)
axes[0, 0].set_title('Balkendiagramm - Gesamtpassagiere nach Flughafen', fontsize=12)
axes[0, 0].set_xlabel('Flughafen')
axes[0, 0].set_ylabel('Gesamtanzahl Passagiere')
axes[0, 0].tick_params(axis='x', rotation=0)
axes[0, 0].grid(axis='y', alpha=0.3)

# Liniendiagramm - Passagiere über Zeit
passengers_by_year.plot(kind='line', ax=axes[0, 1], color=custom_colors[0], linewidth=2, marker='o')
axes[0, 1].set_title('Liniendiagramm - Passagiere über die Jahre', fontsize=12)
axes[0, 1].set_xlabel('Jahr')
axes[0, 1].set_ylabel('Gesamtanzahl Passagiere')
axes[0, 1].grid(alpha=0.3)

# Gruppiertes Balkendiagramm - Flüge nach Flughafen und Verkehrsart
flights_grouped = df_clean.groupby(['C-ZLFE32-0', 'C-ZLFART-0'])['F-ZLFFLUG'].sum().unstack()
flights_grouped.plot(kind='bar', ax=axes[1, 0], color=custom_colors[:2])
axes[1, 0].set_title('Balkendiagramm - Flüge nach Flughafen und Verkehrsart', fontsize=12)
axes[1, 0].set_xlabel('Flughafen')
axes[1, 0].set_ylabel('Anzahl Flüge')
axes[1, 0].legend(['Linienverkehr', 'Charterverkehr'])
axes[1, 0].tick_params(axis='x', rotation=0)
axes[1, 0].grid(axis='y', alpha=0.3)

# Liniendiagramm - Entwicklung pro Flughafen
for i, airport in enumerate(df_clean['C-ZLFE32-0'].unique()):
    subset = df_clean[df_clean['C-ZLFE32-0'] == airport]
    pax_by_year = subset.groupby('Jahr')['F-ZLFPAX'].sum()
    pax_by_year.plot(kind='line', ax=axes[1, 1], label=airport, 
                    linewidth=2, marker='o', color=custom_colors[i % len(custom_colors)])
axes[1, 1].set_title('Liniendiagramm - Passagierentwicklung pro Flughafen', fontsize=12)
axes[1, 1].set_xlabel('Jahr')
axes[1, 1].set_ylabel('Anzahl Passagiere')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAuffälligkeiten:")
print("- VIE dominiert mit Abstand bei Passagierzahlen")
print("- Deutlicher Einbruch in den Jahren 2020-2021 (COVID-19 Pandemie)")
print("- Linienverkehr macht den größten Anteil der Flüge aus")
print("- Alle Flughäfen zeigen ähnliche Trendmuster über die Zeit")
print("- Sichtbare Erholung ab 2022")

In [ ]:
# Zusätzliche Visualisierungen
print("=== ZUSÄTZLICHE ANALYSEN ===")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Monatliche Saisonalität
monthly_pax = df_clean.groupby('Monat')['F-ZLFPAX'].mean()
monthly_pax.plot(kind='bar', ax=axes[0], color=custom_colors)
axes[0].set_title('Durchschnittliche Passagiere nach Monat (Saisonalität)', fontsize=12)
axes[0].set_xlabel('Monat')
axes[0].set_ylabel('Durchschnittliche Passagiere')
axes[0].grid(axis='y', alpha=0.3)

# Richtungsverteilung
direction_counts = df_clean['C-ZLFR63-0'].value_counts()
axes[1].pie(direction_counts, labels=['Abflug', 'Ankunft', 'Transit'], 
           autopct='%1.1f%%', colors=custom_colors[:3], startangle=90)
axes[1].set_title('Verteilung nach Richtung', fontsize=12)

plt.tight_layout()
plt.show()

print("\nAuffälligkeiten:")
print("- Saisonale Schwankungen erkennbar mit Spitzen in Sommermonaten")
print("- Abflug und Ankunft sind etwa gleich verteilt (wie erwartet)")
print("- Transit macht einen kleinen Teil aus")

## Zusammenfassung

### Haupterkenntnisse:

1. **Datenqualität**: Der Datensatz ist vollständig ohne fehlende Werte und ohne fehlerhafte Einträge

2. **Flughäfen**: 
   - VIE (Wien) ist der mit Abstand größte Flughafen
   - Alle anderen Flughäfen haben deutlich geringeres Verkehrsaufkommen

3. **Korrelationen**:
   - Starke Korrelation zwischen Flügen und Passagieren
   - Fracht und Post zeigen unabhängigere Muster

4. **Zeitliche Trends**:
   - Deutlicher COVID-19 Einbruch 2020-2021
   - Saisonale Schwankungen mit Sommerspitzen
   - Erholung ab 2022 sichtbar

5. **Verkehrsarten**:
   - Linienverkehr dominiert über Charterverkehr
   - Unterschiedliche Verteilungsmuster bei Passagierzahlen

### Gespeicherte Dateien:
- `korr.csv`: Bereinigte Daten
- `num.csv`: Numerische Werte
- `norm.csv`: Normalisierte Werte
- `nom.csv`: Nominale Werte